# Meridian Community Bank — LLM-as-Judge Evaluation Notebook

This notebook implements **Steps 7–11** of the LLM Evaluation Lab using `gpt-4o-mini` (via the OpenAI API) paired with deterministic ECOA/FCRA compliance guardrails.

- Set `RUN_MOCK = False` after adding your `OPENAI_API_KEY` to `.env` to execute live API evaluations.
- Keep `RUN_MOCK = True` to test the full pipeline, metrics aggregation, and charts offline.


## Step 7: Setup & Environment


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

from llm_judge_evaluation import (
    DEFAULT_MODEL,
    JUDGE_SYSTEM_PROMPT,
    load_test_dataset,
    run_evaluation_pipeline,
    print_summary_report,
)

RUN_MOCK = True  # Change to False to run live against OpenAI API (requires .env)
MODEL_NAME = DEFAULT_MODEL
print(f'Configured model: {MODEL_NAME} | Mock mode: {RUN_MOCK}')


## Step 8 & Step 9: Inspect Judge Prompt & 5-Case Test Dataset


In [ ]:
dataset = load_test_dataset()
print(f'Loaded {len(dataset)} test cases:')
for tc in dataset:
    print(f"  - {tc['id']}: {tc['title']} [{tc['difficulty']}]")

print('\nJudge Prompt Preview (first 400 chars):')
print(JUDGE_SYSTEM_PROMPT[:400] + '...')


## Step 10: Run Evaluation & Collect Metrics


In [ ]:
results = run_evaluation_pipeline(
    mock_mode=RUN_MOCK,
    model=MODEL_NAME,
    output_path=Path('evaluation_results.json'),
)
print_summary_report(results)


## Step 11: Analyze & Visualize Results


In [ ]:
items = results['individual_results']
case_ids = [r['id'] for r in items]
scores = [r['score'] for r in items]
criteria_rates = results['aggregate_metrics']['criteria_pass_rates']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

colors = ['#2E7D32' if s >= 4 else ('#ED6C02' if s == 3 else '#C62828') for s in scores]
axes[0].bar(case_ids, scores, color=colors, edgecolor='black', alpha=0.88)
axes[0].set_ylim(0, 5.5)
axes[0].set_title('LLM-as-Judge Score by Test Case (1-5)')
axes[0].set_ylabel('Score')
axes[0].axhline(4.0, color='gray', linestyle='--', linewidth=1, label='Production Target (4.0)')
axes[0].legend()
for idx, val in enumerate(scores):
    axes[0].text(idx, val + 0.12, str(val), ha='center', fontweight='bold')

crit_names = [k.replace('_', '\n') for k in criteria_rates.keys()]
crit_vals = [v * 100 for v in criteria_rates.values()]
axes[1].barh(crit_names, crit_vals, color='#1565C0', edgecolor='black', alpha=0.88)
axes[1].set_xlim(0, 110)
axes[1].set_title('Rubric Criteria Pass Rates (%)')
axes[1].set_xlabel('Pass Rate (%)')
for idx, val in enumerate(crit_vals):
    axes[1].text(val + 2, idx, f'{val:.0f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()
